In [ ]:
#Routr project

#The goal is to design an app that plans a route through Madison/Boulder for you based off of bike lanes using Dijkstra's
#Or a Euler trail for certain destinations for sightseeing.

In [1]:
import os
# Replace the path with your actual proj directory


# Now you can import your libraries
import geopandas
import osmnx as ox
import networkx as nx
import numpy as np

# os.environ['PROJ_LIB'] = r'C:/Users/monke/miniconda3/envs/routr_app/Library/share/proj'

In [ ]:
# import heapq as hq

In [2]:
bike_graph = ox.graph_from_place(query = 'Madison, Wisconsin', network_type= 'bike')

In [3]:
start_node = 11143855834
end_node = 1179967122

In [ ]:
def find_shorest_neighbor_lengths(start_node):
    """Iterates over the neighboring nodes of the input start_node to find their edge lengths.

    Args:
        start_node (int): the beginning node id 
    """
    
    neighbor_nodes = list(bike_graph.neighbors(start_node))

    for neighbor in neighbor_nodes:
        neighbor_edges_dict = bike_graph[start_node][neighbor]
        for edge_key in neighbor_edges_dict:
            edge = neighbor_edges_dict[edge_key]
            edge_length = edge['length']
            return edge_length

In [ ]:
def find_shorest_neighbor_edge(start_node):
    """Iterates over the neighboring nodes of the input start_node to find their edge lengths.

    Args:
        start_node (int): the beginning node id 
    """
    
    neighbor_nodes = list(bike_graph.neighbors(start_node))

    for neighbor in neighbor_nodes:
        neighbor_edges_dict = bike_graph[start_node][neighbor]
        closest_neighbor_node = min(neighbor_edges_dict, key = neighbor_edges_dict.get)
        return closest_neighbor_node

In [ ]:
def find_shortest_path(start_node, end_node):
    node_distances = {node : float('inf') for node in list(bike_graph.nodes)}
    node_distances[start_node] = 0
    previous_nodes = dict()
    # unvisited_nodes = list(bike_graph.nodes)
    unvisited_nodes = set(bike_graph.nodes)
    shortest_path = list()
    min_distance_so_far = float("inf")



    while len(unvisited_nodes) != 0:
    #     potent_closest_node = min(node_distances, key = node_distances.get) 
        for node, dist in node_distances.items():
            if ((node in unvisited_nodes) & (dist < min_distance_so_far)):
                closest_node = node
                min_distance_so_far = dist

        if closest_node == end_node:
    #     if previous_nodes[closest_node] != float('inf') or closest_node == start_node:
            while previous_nodes[closest_node]:
                closest_node = previous_nodes[closest_node]
                shortest_path.append(closest_node)
            #     closest_node = closest_node
            return shortest_path[::-1]
            
    #     if closest_node in unvisited_nodes:
    #         print(f"{closest_node} in unvisited nodes")
    #     unvisited_nodes.remove(closest_node)
        else:
            for possible_node in bike_graph[closest_node]:
                    edge_option = find_shorest_neighbor_edge(possible_node)
                    edge_length = bike_graph[closest_node][possible_node][edge_option]['length'] #NEED TO LOOK AT EACH EDGE WITHIN THE PREVIOUS NODES
                    
                    alt_path_length = node_distances[closest_node] + edge_length
                    if node_distances[possible_node] > alt_path_length:
                        node_distances[possible_node] = alt_path_length
                        previous_nodes[possible_node] = closest_node


    

# return shortest_path


    

In [ ]:
find_shortest_path(start_node, end_node)

In [ ]:
find_shorest_neighbor_edge(start_node)

In [ ]:
def find_shortest_path_queue(start_node, end_node):
    node_distances = {node : float('inf') for node in list(bike_graph.nodes)}
    node_distances[start_node] = 0
    previous_nodes = dict()
    unvisited_nodes = set(bike_graph.nodes)
    shortest_path = list()
    min_distance_so_far = float("inf")



    while len(unvisited_nodes) != 0: 
        for node, dist in node_distances.items():
            if ((node in unvisited_nodes) & (dist < min_distance_so_far)):
                closest_node = node
                min_distance_so_far = dist

        if closest_node == end_node:
            while previous_nodes[closest_node]:
                closest_node = previous_nodes[closest_node]
                shortest_path.append(closest_node)
            return shortest_path[::-1]
            
        else:
            for possible_node in bike_graph[closest_node]:
                    edge_option = find_shorest_neighbor_edge(possible_node)
                    edge_length = bike_graph[closest_node][possible_node][edge_option]['length'] #NEED TO LOOK AT EACH EDGE WITHIN THE PREVIOUS NODES, currently assuming 0
                    
                    alt_path_length = node_distances[closest_node] + edge_length
                    if node_distances[possible_node] > alt_path_length:
                        node_distances[possible_node] = alt_path_length
                        previous_nodes[possible_node] = closest_node


    

# return shortest_path


    

In [ ]:
# find_shortest_path(start_node, end_node)

In [67]:
node_distances = {node : float('inf') for node in list(bike_graph.nodes)}
node_distances[start_node] = 0
previous_nodes = dict()
unvisited_nodes = set(bike_graph.nodes)
shortest_path = list()


while len(unvisited_nodes) != 0: 
    min_distance_so_far = float("inf")
    for node, dist in node_distances.items():
        if ((node in unvisited_nodes) & (dist < min_distance_so_far)):
            closest_node = node
            print(f'closest_node updated to {closest_node}')
            if closest_node != start_node:
                min_distance_so_far = dist
                

    if closest_node == end_node:
        while previous_nodes[closest_node]:
            closest_node = previous_nodes[closest_node]
            shortest_path.append(closest_node)
            shortest_path =  shortest_path[::-1]
        
    else:
        unvisited_nodes.remove(closest_node)
        for possible_node in bike_graph[closest_node]:
            if possible_node in unvisited_nodes:

                # edge_option = find_shorest_neighbor_edge(closest_node, possible_node)
                edge_option = min(bike_graph[closest_node][possible_node], key = bike_graph[closest_node][possible_node].get)
                
                edge_length = bike_graph[closest_node][possible_node][edge_option]['length'] #NEED TO LOOK AT EACH EDGE WITHIN THE PREVIOUS NODES, currently assuming 0
                alt_path_length = node_distances[closest_node] + edge_length
                if node_distances[possible_node] > alt_path_length:
                    node_distances[possible_node] = alt_path_length
                    previous_nodes[possible_node] = closest_node
            else:
                continue


#Issue currently is:
# The minimum distance is always 0, so then the closest node is always the start_node
# Second part of the & statement will always be false
# How to avoid this?
# New iteration/loop after the first node 
# Don't update the minimum distance for the first node?

closest_node updated to 11143855834
closest_node updated to 11143855833
closest_node updated to 3101275546
closest_node updated to 6223342038
closest_node updated to 1583851858
closest_node updated to 3101275546
closest_node updated to 452858739
closest_node updated to 1583851858
closest_node updated to 452858739
closest_node updated to 3101275542
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 3945880904
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 6223342062
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 3945880900
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 452858693
closest_node updated to 53596637
closest_node updated to 452858739
closest_node updated to 6223351255
closest_node updated to 53596637
closest_node updated to 452858739
closest_node updated to 8109848400
closest_node updated to 5359663

TypeError: '<' not supported between instances of 'dict' and 'dict'

In [ ]:
node_distances = {node : float('inf') for node in list(bike_graph.nodes)}
node_distances[start_node] = 0
previous_nodes = dict()
unvisited_nodes = set(bike_graph.nodes)
shortest_path = list()
# min_distance_so_far = float("inf")



while len(unvisited_nodes) != 0:
    min_distance_so_far = float("inf")
    closest_node = None
    for node, dist in node_distances.items():
        if ((node in unvisited_nodes) & (dist < min_distance_so_far)):
            closest_node = node
            min_distance_so_far = dist
            print(f'closest_node updated to {closest_node}')
            # if closest_node != start_node:
            #     min_distance_so_far = dist
            
                


    if closest_node == end_node:
        while previous_nodes[closest_node]:
            closest_node = previous_nodes[closest_node]
            shortest_path.append(closest_node)
            shortest_path =  shortest_path[::-1]
            #Something is wrong here...

    if closest_node == None:
        break
        

    unvisited_nodes.remove(closest_node)
    for possible_node in bike_graph[closest_node]:
        if possible_node in unvisited_nodes:

            # edge_option = find_shorest_neighbor_edge(closest_node, possible_node)
            # edge_option = min(bike_graph[closest_node][possible_node], key = bike_graph[closest_node][possible_node].get)
            edge_option = min(bike_graph[closest_node][possible_node], key = lambda edge_id: bike_graph[closest_node][possible_node].get(edge_id)['length'] )
            edge_length = bike_graph[closest_node][possible_node][edge_option]['length'] #NEED TO LOOK AT EACH EDGE WITHIN THE PREVIOUS NODES, currently assuming 0
            alt_path_length = node_distances[closest_node] + edge_length
            if node_distances[possible_node] > alt_path_length:
                node_distances[possible_node] = alt_path_length
                previous_nodes[possible_node] = closest_node
        else:
            continue
        


#Issue currently is:
# The minimum distance is always 0, so then the closest node is always the start_node
# Second part of the & statement will always be false
# How to avoid this?
# New iteration/loop after the first node 
# Don't update the minimum distance for the first node

closest_node updated to 11143855834
closest_node updated to 11143855833
closest_node updated to 3101275546
closest_node updated to 6223342038
closest_node updated to 1583851858
closest_node updated to 3101275546
closest_node updated to 452858739
closest_node updated to 1583851858
closest_node updated to 452858739
closest_node updated to 3101275542
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 3945880904
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 6223342062
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 3945880900
closest_node updated to 452858739
closest_node updated to 3101275543
closest_node updated to 452858693
closest_node updated to 53596637
closest_node updated to 452858739
closest_node updated to 6223351255
closest_node updated to 53596637
closest_node updated to 452858739
closest_node updated to 8109848400
closest_node updated to 5359663

KeyError: 11143855834

In [134]:
shortest_path

[11143855834,
 3101275546,
 3101275543,
 53596637,
 53596635,
 4055677013,
 452265822,
 7451680639,
 7451680644,
 7811575601,
 37932840,
 6915282275,
 53377506,
 53461820,
 11990782715,
 1180005830,
 3607816606,
 3035567517,
 3894523242,
 3035567511,
 53461798,
 2987141996,
 53596818,
 1345424866,
 53607075,
 13003579526,
 53668988,
 2410237412,
 53668981,
 1179956577,
 2859307973,
 7313833034,
 13056348952,
 13056348953,
 7313833033,
 1179956515,
 1179981715,
 2410237418,
 53668984,
 53607068,
 1533633330,
 1859358892,
 1345424868,
 53456049,
 53461796,
 2848493174,
 1857601486,
 2848493172,
 2848493171,
 3607816602,
 53461816,
 53461818,
 3608310539,
 8419593265,
 7811575594,
 7185777721,
 7451680651,
 7451680649,
 4214318025,
 4055677012,
 53596634,
 3033922199,
 452858693,
 3101275542,
 11143855833]

In [135]:
end_node

1179967122

In [42]:
min_distance_so_far

np.float64(63.18490260454027)

In [ ]:
neighbor_nodes = list(bike_graph.neighbors(start_node))

for neighbor in neighbor_nodes:
    neighbor_edges_dict = bike_graph[start_node][neighbor]
    closest_neighbor_node = min(neighbor_edges_dict, key = neighbor_edges_dict.get)
    output =  closest_neighbor_node

In [ ]:
class CyclewayGraph_digraph:
    
    def __init__(self, city, state):
        self.id = f"{city}, {state}"

        self.city = city
        self.state = state

        bike_data = ox.graph_from_place(query = self.id, network_type= 'bike')

        self.nodes, self.edges = bike_data.nodes, bike_data

        cycleway_edges_unsort = self.bike_gdf_edges.loc[self.bike_gdf_edges.loc[:,'highway']=='cycleway']

        self.cyc_edges = cycleway_edges_unsort.sort_index()

        self.all_nodes = set(self.bike_gdf_nodes.index)

    def shortest_path(self, start, end):

        unvisited_nodes = {node_id : float('inf') for node_id in self.all_nodes} #gather all nodes in a dictionary 
        unvisited_nodes[start] = 0 #set starting node length to 0

        prev_nodes = {node_id : np.nan for node_id in self.all_nodes}

        current_node = start #Should be unnecessary

        while len(unvisited_nodes) != 0:
            
            shortest_next_edge_length =  self.cyc_edges.loc[current_node,'length'].min() #somethings wrong here.
            
            minimum_dist_node = min(unvisited_nodes, key = unvisited_nodes.get)

            removed_len  = unvisited_nodes.pop(minimum_dist_node)
            for next_node in self.cyc_edges.loc[current_node].index.get_level_values('v'):
                new_length = removed_len + self.cyc_edges.loc[(13107812002,13107783597), 'length'].item()
                if new_length < unvisited_nodes[next_node]:
                    unvisited_nodes[next_node] = new_length
                    prev_nodes[next_node] = current_node

            return unvisited_nodes



In [132]:
min(bike_graph[closest_node][possible_node], key = lambda edge_id: bike_graph[closest_node][possible_node].get(edge_id)['length'] )

0

In [111]:
for edge_id in bike_graph[closest_node][possible_node]:
    print(bike_graph[closest_node][possible_node].get(edge_id)['length'])

32.28200188952122
66.03930120872724


In [131]:
for edge_id in bike_graph[closest_node][possible_node]:
    print(bike_graph[closest_node][possible_node][edge_id]['length'])

32.28200188952122
66.03930120872724


In [130]:
bike_graph[closest_node]

AdjacencyView({8402901945: {0: {'osmid': 722472492, 'highway': 'service', 'service': 'parking_aisle', 'oneway': False, 'reversed': False, 'length': np.float64(32.28200188952122)}, 1: {'osmid': 904814901, 'highway': 'service', 'service': 'parking_aisle', 'oneway': False, 'reversed': False, 'length': np.float64(66.03930120872724), 'geometry': <LINESTRING (-89.427 43.077, -89.427 43.077, -89.427 43.078, -89.427 43.078)>}}, 6776846629: {0: {'osmid': 722472492, 'highway': 'service', 'service': 'parking_aisle', 'oneway': False, 'reversed': True, 'length': np.float64(22.351632965213305), 'geometry': <LINESTRING (-89.427 43.077, -89.427 43.077, -89.427 43.077)>}}})

In [ ]:
class CyclewayGraph:
    
    def __init__(self, city, state):
        self.id = f"{city}, {state}"

        self.city = city
        self.state = state

        bike_data = ox.graph_from_place(query = self.id, network_type= 'bike')

        self.bike_gdf_nodes, self.bike_gdf_edges = ox.graph_to_gdfs(bike_data)

        cycleway_edges_unsort = self.bike_gdf_edges.loc[self.bike_gdf_edges.loc[:,'highway']=='cycleway']

        self.cyc_edges = cycleway_edges_unsort.sort_index()

        self.all_nodes = set(self.bike_gdf_nodes.index)

    def shortest_path(self, start, end):

        unvisited_nodes = {node_id : float('inf') for node_id in self.all_nodes} #gather all nodes in a dictionary 
        unvisited_nodes[start] = 0 #set starting node length to 0

        prev_nodes = {node_id : np.nan for node_id in self.all_nodes}

        current_node = start #Should be unnecessary

        while len(unvisited_nodes) != 0:
            
            shortest_next_edge_length =  self.cyc_edges.loc[current_node,'length'].min() #somethings wrong here.
            
            minimum_dist_node = min(unvisited_nodes, key = unvisited_nodes.get)

            removed_len  = unvisited_nodes.pop(minimum_dist_node)
            for next_node in self.cyc_edges.loc[current_node].index.get_level_values('v'):
                new_length = removed_len + self.cyc_edges.loc[(13107812002,13107783597), 'length'].item()
                if new_length < unvisited_nodes[next_node]:
                    unvisited_nodes[next_node] = new_length
                    prev_nodes[next_node] = current_node

            return unvisited_nodes



In [ ]:
msn_bike = CyclewayGraph('Madison', 'Wisconsin')

In [ ]:
bike_network = msn_bike.bike_gdf_edges.explore()

In [ ]:
msn_bike.bike_gdf_edges.loc[13107812002].explore(
    m = bike_network,
    color = 'red',
    name = 'start_edge'
)

In [ ]:
# (msn_bike.cyc_edges.loc[:,'osmid'] == 1201999277).any()
type(msn_bike.cyc_edges.loc[(53281461,3589292398),'osmid'].item()[0])

In [ ]:
msn_bike[msn_bike.cyc_edges.loc[:,'osmid'] == 1201999277]

In [ ]:
# msn_bike.cyc_edges.loc[(13107812002,13107812003)]
msn_bike.cyc_edges.loc[(13107812002,13107783597), 'length'].item()

In [ ]:
unvisited_nodes = msn_bike.shortest_path(13107812002, 13107812003)

In [ ]:
unvisited_nodes[13107812002]

In [ ]:
unvisited_nodes[min( unvisited_nodes, key = unvisited_nodes.get)]

In [ ]:
# np.float(msn_bike.cyc_edges.loc[13107812002,'length'])
(msn_bike.cyc_edges.loc[13107812002,'length'].item())

In [ ]:
msn_cycleway_edges = bike_gdf_edges.loc[bike_gdf_edges.loc[:,'highway']=='cycleway']

In [ ]:
# test_edge = find_shortest_path(13107812002, 13107812003)

In [ ]:
# test_edge = find_shortest_path(13107812002, 13107812003)
visited_nodes = {}

point_a = 13107812002

point_b = 13107812003

for next_node in msn_cycleway_edges.loc[point_a].index: #iterate over each node point_a is connected to 
    #iterate over all edges connected to a and find the shortest


    pot_edge = msn_cycleway_edges.loc[next_node[0]] #use nodes from point_a to find which edges they connect to 
    print(pot_edge.loc[:,'length'])
    edge_length = pot_edge.loc[:,'length']
    # return edge_length

In [ ]:
#Drafting shortest path function:

point_a = 13107812002

current_node = point_a

msn_bike_data = ox.graph_from_place(query = 'Madison, Wisconsin', network_type= 'bike')

bike_gdf_nodes, bike_gdf_edges = ox.graph_to_gdfs(msn_bike_data)

msn_cycleway_edges_unsort = bike_gdf_edges.loc[bike_gdf_edges.loc[:,'highway']=='cycleway']

msn_cycleway_edges = msn_cycleway_edges_unsort.sort_index()

all_nodes_set = set(bike_gdf_nodes.index)

unvisited_nodes = {node_id : float('inf') for node_id in all_nodes_set} #gather all nodes in a dictionary 

unvisited_nodes[point_a] = 0 #set starting node length to 0

visited_nodes = {}

In [ ]:
# current_node == point_a
current_node is point_a

In [ ]:
#Drafting actual function part:

for next_node in msn_cycleway_edges.loc[current_node].index: #iterate over each node current_node is connected to 
    #Displays what vertices are next beyond current_node
    all_next_nodes = msn_cycleway_edges.loc[current_node].index.get_level_values(0)
    
    # all_next_node_lengths = msn_cycleway_edges.loc[(current_node, all_next_nodes[0]), 'length'] #attempt at finding all lengths, may not be necessary?

    for idx, node in enumerate(all_next_nodes):
        unvisited_nodes[node] = msn_cycleway_edges.loc[(current_node, node), 'length'][0] #uses an np float64, necessary or change to py float?

    #IS THIS UPDATING THE VALUES PROPERLY?

    next_edges_lengths =  msn_cycleway_edges.loc[current_node,'length']
    
    shortest_next_edge_length =  msn_cycleway_edges.loc[current_node,'length'].min()
    
    closest_node = msn_cycleway_edges.loc[current_node,'length'].idxmin()[0]

    if shortest_next_edge_length < unvisited_nodes[closest_node]:
        unvisited_nodes[closest_node] = shortest_next_edge_length

    


In [ ]:
#I need to get all vertices in the graph first
# Create a separate dataframe to record all edges
# Add all  

In [ ]:
#Find two nodes to travel between for the function.
#Can index the edges by starting vertex to find other options,
#Use this (with recursion?)/while loop to implement dijkstra's
#Do I need the nodes at all?
# msn_cycleway_edges.loc[13107812002]

# msn_cycleway_edges.loc[13107812003]
# msn_cycleway_edges.loc[13107783584]


In [ ]:
#The end goal is to return a path (no repeated vertices or edges) from a point A to a point B.